In [1]:
from pathlib import Path
import json
import math
import traceback

import numpy as np
import pandas as pd

from Bio.PDB import PDBParser, MMCIFParser
from Bio.PDB.Polypeptide import is_aa
from Bio.PDB.vectors import calc_dihedral

try:
    from Bio.PDB.DSSP import DSSP
    DSSP_AVAILABLE = True
except Exception:
    DSSP_AVAILABLE = False

In [2]:
DATA_DIR = Path("data")

PDB_FILES_DIR = DATA_DIR / "raw" / "pdb_files"
CIF_FILES_DIR = DATA_DIR / "raw" / "cif_files"

PDB_IDS_JSON_PATH = DATA_DIR / "raw" / "filtered_pdb_ids.json"

PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

N_BINS = 36
BIN_SIZE_DEGREES = 10
IGNORE_INDEX = -100

print("DSSP import available:", DSSP_AVAILABLE)

DSSP import available: True


In [3]:
with open(PDB_IDS_JSON_PATH, "r") as f:
    pdb_ids = json.load(f)

pdb_ids = [pdb_id.upper() for pdb_id in pdb_ids]

print("Number of PDB IDs:", len(pdb_ids))
print(pdb_ids[:10])

Number of PDB IDs: 6108
['2BO5', '2BW2', '2IVW', '4ME2', '4MEI', '4MFI', '4MFJ', '4MGS', '4MHP', '4MPL']


In [4]:
def find_structure_file(pdb_id: str):
    pdb_id_lower = pdb_id.lower()

    candidate_paths = [
        CIF_FILES_DIR / f"{pdb_id_lower}.cif",
        PDB_FILES_DIR / f"pdb{pdb_id_lower}.ent",
    ]

    for path in candidate_paths:
        if path.exists():
            return path

    return None

In [6]:
def load_structure(pdb_id: str, structure_path: Path):
    suffix = structure_path.suffix.lower()

    if suffix in {".cif"}:
        parser = MMCIFParser(QUIET=True)
    elif suffix in {".ent"}:
        parser = PDBParser(QUIET=True)
    else:
        raise ValueError(f"Unsupported structure format: {structure_path}")

    return parser.get_structure(pdb_id, structure_path)

In [7]:
STANDARD_AA_3 = {
    "ALA", "ARG", "ASN", "ASP", "CYS",
    "GLN", "GLU", "GLY", "HIS", "ILE",
    "LEU", "LYS", "MET", "PHE", "PRO",
    "SER", "THR", "TRP", "TYR", "VAL",
}

In [8]:
CHI1_ATOMS = {
    "ARG": ("N", "CA", "CB", "CG"),
    "ASN": ("N", "CA", "CB", "CG"),
    "ASP": ("N", "CA", "CB", "CG"),
    "CYS": ("N", "CA", "CB", "SG"),
    "GLN": ("N", "CA", "CB", "CG"),
    "GLU": ("N", "CA", "CB", "CG"),
    "HIS": ("N", "CA", "CB", "CG"),
    "ILE": ("N", "CA", "CB", "CG1"),
    "LEU": ("N", "CA", "CB", "CG"),
    "LYS": ("N", "CA", "CB", "CG"),
    "MET": ("N", "CA", "CB", "CG"),
    "PHE": ("N", "CA", "CB", "CG"),
    "PRO": ("N", "CA", "CB", "CG"),
    "SER": ("N", "CA", "CB", "OG"),
    "THR": ("N", "CA", "CB", "OG1"),
    "TRP": ("N", "CA", "CB", "CG"),
    "TYR": ("N", "CA", "CB", "CG"),
    "VAL": ("N", "CA", "CB", "CG1"),
}

In [9]:
AA_PROPERTIES = {
    "ALA": ("nonpolar", "hydrophobic", "small", "aliphatic", "nonpolar_aliphatic", "neutral", "none"),
    "ARG": ("polar", "hydrophilic", "large", "basic", "positive", "positive", "donor"),
    "ASN": ("polar", "hydrophilic", "medium", "amide", "polar_uncharged", "neutral", "donor_acceptor"),
    "ASP": ("polar", "hydrophilic", "medium", "acidic", "negative", "negative", "acceptor"),
    "CYS": ("polar", "hydrophobic", "small", "sulfur", "special", "neutral", "donor"),
    "GLN": ("polar", "hydrophilic", "large", "amide", "polar_uncharged", "neutral", "donor_acceptor"),
    "GLU": ("polar", "hydrophilic", "large", "acidic", "negative", "negative", "acceptor"),
    "GLY": ("nonpolar", "neutral", "very_small", "special", "special", "neutral", "none"),
    "HIS": ("polar", "hydrophilic", "large", "basic_aromatic", "positive_weak", "positive_weak", "donor_acceptor"),
    "ILE": ("nonpolar", "hydrophobic", "large", "aliphatic", "nonpolar_aliphatic", "neutral", "none"),
    "LEU": ("nonpolar", "hydrophobic", "large", "aliphatic", "nonpolar_aliphatic", "neutral", "none"),
    "LYS": ("polar", "hydrophilic", "large", "basic", "positive", "positive", "donor"),
    "MET": ("nonpolar", "hydrophobic", "large", "sulfur", "nonpolar_sulfur", "neutral", "none"),
    "PHE": ("nonpolar", "hydrophobic", "large", "aromatic", "aromatic", "neutral", "none"),
    "PRO": ("nonpolar", "hydrophobic", "medium", "cyclic", "special", "neutral", "none"),
    "SER": ("polar", "hydrophilic", "small", "hydroxyl", "polar_uncharged", "neutral", "donor_acceptor"),
    "THR": ("polar", "hydrophilic", "medium", "hydroxyl", "polar_uncharged", "neutral", "donor_acceptor"),
    "TRP": ("nonpolar", "hydrophobic", "large", "aromatic", "aromatic", "neutral", "donor"),
    "TYR": ("polar", "hydrophobic", "large", "aromatic_hydroxyl", "aromatic_polar", "neutral", "donor_acceptor"),
    "VAL": ("nonpolar", "hydrophobic", "medium", "aliphatic", "nonpolar_aliphatic", "neutral", "none"),
}

PROPERTY_COLUMNS = [
    "polarity_group",
    "hydropathy_group",
    "volume_group",
    "chemical_group",
    "physicochemical_group",
    "charge_group",
    "hydrogen_donor_acceptor_group",
]

In [10]:
def add_amino_acid_properties(df: pd.DataFrame):
    df = df.copy()

    for i, column in enumerate(PROPERTY_COLUMNS):
        df[column] = df["resname"].map(lambda aa: AA_PROPERTIES[aa][i])

    return df

In [11]:
def normalize_angle_degrees(angle):
    if angle is None or pd.isna(angle):
        return np.nan

    return float(angle) % 360.0

In [12]:
def angle_to_10deg_bin(angle):
    angle = normalize_angle_degrees(angle)

    if pd.isna(angle):
        return IGNORE_INDEX

    return int(angle // BIN_SIZE_DEGREES)

In [13]:
def angle_to_sin_cos(angle):
    if angle is None or pd.isna(angle):
        return 0.0, 0.0

    angle_rad = math.radians(float(angle))

    return math.sin(angle_rad), math.cos(angle_rad)

In [14]:
def parse_residue_id(residue):
    hetero_flag, residue_number, insertion_code = residue.id
    insertion_code = insertion_code.strip()

    return hetero_flag, residue_number, insertion_code

In [15]:
def get_atom_vector(residue, atom_name):
    if atom_name not in residue:
        return None

    return residue[atom_name].get_vector()

In [16]:
def get_atom_coord(residue, atom_name):
    if atom_name not in residue:
        return None

    return residue[atom_name].get_coord()


def compute_dihedral_degrees(residue, atom_names):
    vectors = []

    for atom_name in atom_names:
        vector = get_atom_vector(residue, atom_name)

        if vector is None:
            return np.nan

        vectors.append(vector)

    angle_rad = calc_dihedral(*vectors)
    angle_deg = math.degrees(angle_rad)

    return normalize_angle_degrees(angle_deg)

In [17]:
def compute_phi(prev_residue, residue):
    if prev_residue is None:
        return np.nan

    c_prev = get_atom_vector(prev_residue, "C")
    n = get_atom_vector(residue, "N")
    ca = get_atom_vector(residue, "CA")
    c = get_atom_vector(residue, "C")

    if c_prev is None or n is None or ca is None or c is None:
        return np.nan

    angle_rad = calc_dihedral(c_prev, n, ca, c)
    angle_deg = math.degrees(angle_rad)

    return normalize_angle_degrees(angle_deg)


def compute_psi(residue, next_residue):
    if next_residue is None:
        return np.nan

    n = get_atom_vector(residue, "N")
    ca = get_atom_vector(residue, "CA")
    c = get_atom_vector(residue, "C")
    n_next = get_atom_vector(next_residue, "N")

    if n is None or ca is None or c is None or n_next is None:
        return np.nan

    angle_rad = calc_dihedral(n, ca, c, n_next)
    angle_deg = math.degrees(angle_rad)

    return normalize_angle_degrees(angle_deg)

In [18]:
def get_cb_like_coord(residue, resname):
    if resname == "GLY":
        return None

    return get_atom_coord(residue, "CB")

In [19]:
def compute_ca_cb_direction(residue, resname):
    ca_coord = get_atom_coord(residue, "CA")
    cb_coord = get_cb_like_coord(residue, resname)

    if ca_coord is None or cb_coord is None:
        return 0.0, 0.0, 0.0, 0

    vector = cb_coord - ca_coord
    norm = np.linalg.norm(vector)

    if norm < 1e-8:
        return 0.0, 0.0, 0.0, 0

    vector = vector / norm

    return float(vector[0]), float(vector[1]), float(vector[2]), 1

In [20]:
def is_chain_break_prev(prev_residue, residue):
    if prev_residue is None:
        return 1

    _, prev_number, _ = parse_residue_id(prev_residue)
    _, residue_number, _ = parse_residue_id(residue)

    return int(residue_number - prev_number != 1)

In [21]:
def is_chain_break_next(residue, next_residue):
    if next_residue is None:
        return 1

    _, residue_number, _ = parse_residue_id(residue)
    _, next_number, _ = parse_residue_id(next_residue)

    return int(next_number - residue_number != 1)

In [22]:
def compute_dssp_dict(model, structure_path):
    if not DSSP_AVAILABLE:
        return {}

    try:
        dssp = DSSP(model, str(structure_path))
    except Exception as e:
        return {}

    dssp_dict = {}

    for key in dssp.keys():
        chain_id, residue_id = key
        dssp_values = dssp[key]

        dssp_ss = dssp_values[2]
        dssp_asa = dssp_values[3]
        dssp_phi = dssp_values[4]
        dssp_psi = dssp_values[5]

        if dssp_ss == "-":
            dssp_ss = "C"

        dssp_dict[(chain_id, residue_id)] = {
            "dssp_ss": str(dssp_ss),
            "dssp_asa": float(dssp_asa),
            "dssp_phi": float(dssp_phi),
            "dssp_psi": float(dssp_psi),
            "dssp_has": 1,
        }

    return dssp_dict

In [23]:
DEFAULT_DSSP_INFO = {
    "dssp_ss": "UNK",
    "dssp_asa": 0.0,
    "dssp_phi": 0.0,
    "dssp_psi": 0.0,
    "dssp_has": 0,
}

In [24]:
def extract_residues_from_structure(pdb_id: str, structure_path: Path):
    structure = load_structure(pdb_id, structure_path)

    rows = []

    model = structure[0]
    dssp_dict = compute_dssp_dict(model, structure_path)

    for chain in model:
        chain_id = chain.id

        valid_residues = []

        for residue in chain:
            hetero_flag, residue_number, insertion_code = parse_residue_id(residue)

            if hetero_flag.strip() != "":
                continue

            resname = residue.get_resname().strip().upper()

            if resname not in STANDARD_AA_3:
                continue

            if not is_aa(residue, standard=True):
                continue

            valid_residues.append(residue)

        for i, residue in enumerate(valid_residues):
            prev_residue = valid_residues[i - 1] if i > 0 else None
            next_residue = valid_residues[i + 1] if i + 1 < len(valid_residues) else None

            _, residue_number, insertion_code = parse_residue_id(residue)
            resname = residue.get_resname().strip().upper()

            chi1_angle = np.nan

            if resname in CHI1_ATOMS:
                chi1_angle = compute_dihedral_degrees(
                    residue=residue,
                    atom_names=CHI1_ATOMS[resname],
                )

            if pd.isna(chi1_angle):
                chi1_bin = IGNORE_INDEX
                target_mask = 0
            else:
                chi1_bin = angle_to_10deg_bin(chi1_angle)
                target_mask = 1

            phi_angle = compute_phi(prev_residue, residue)
            psi_angle = compute_psi(residue, next_residue)

            phi_sin, phi_cos = angle_to_sin_cos(phi_angle)
            psi_sin, psi_cos = angle_to_sin_cos(psi_angle)

            ca_coord = get_atom_coord(residue, "CA")

            if ca_coord is None:
                ca_x, ca_y, ca_z = np.nan, np.nan, np.nan
                has_ca = 0
            else:
                ca_x = float(ca_coord[0])
                ca_y = float(ca_coord[1])
                ca_z = float(ca_coord[2])
                has_ca = 1

            cb_coord = get_cb_like_coord(residue, resname)

            if cb_coord is None:
                cb_x, cb_y, cb_z = np.nan, np.nan, np.nan
                has_cb = 0
            else:
                cb_x = float(cb_coord[0])
                cb_y = float(cb_coord[1])
                cb_z = float(cb_coord[2])
                has_cb = 1

            ca_cb_x, ca_cb_y, ca_cb_z, has_ca_cb_direction = compute_ca_cb_direction(
                residue=residue,
                resname=resname,
            )

            dssp_info = dssp_dict.get((chain_id, residue.id), DEFAULT_DSSP_INFO)

            rows.append({
                "pdb_id": pdb_id.upper(),
                "chain_id": chain_id,
                "residue_number": residue_number,
                "insertion_code": insertion_code,
                "resname": resname,

                "chi1_angle": chi1_angle,
                "chi1_bin_10deg": chi1_bin,
                "target_mask": target_mask,

                "phi_angle": phi_angle,
                "psi_angle": psi_angle,
                "phi_sin": phi_sin,
                "phi_cos": phi_cos,
                "psi_sin": psi_sin,
                "psi_cos": psi_cos,
                "has_phi": int(not pd.isna(phi_angle)),
                "has_psi": int(not pd.isna(psi_angle)),

                "ca_x": ca_x,
                "ca_y": ca_y,
                "ca_z": ca_z,
                "has_ca": has_ca,

                "cb_x": cb_x,
                "cb_y": cb_y,
                "cb_z": cb_z,
                "has_cb": has_cb,

                "ca_cb_x": ca_cb_x,
                "ca_cb_y": ca_cb_y,
                "ca_cb_z": ca_cb_z,
                "has_ca_cb_direction": has_ca_cb_direction,

                "chain_break_prev": is_chain_break_prev(prev_residue, residue),
                "chain_break_next": is_chain_break_next(residue, next_residue),

                "dssp_ss": dssp_info["dssp_ss"],
                "dssp_asa": dssp_info["dssp_asa"],
                "dssp_phi": dssp_info["dssp_phi"],
                "dssp_psi": dssp_info["dssp_psi"],
                "dssp_has": dssp_info["dssp_has"],
            })

    return pd.DataFrame(rows)

In [28]:
def collect_full_feature_dataset_to_csv(pdb_ids, output_path, chunk_size=100):
    output_path = Path(output_path)

    missing_files = []
    failed_structures = []

    buffer = []
    is_first_write = True

    if output_path.exists():
        output_path.unlink()

    for i, pdb_id in enumerate(pdb_ids):
        print(f"[{i + 1}/{len(pdb_ids)}] {pdb_id}", flush=True)

        structure_path = find_structure_file(pdb_id)

        if structure_path is None:
            missing_files.append(pdb_id)
            print("  missing file", flush=True)
            continue

        try:
            df_one = extract_residues_from_structure(
                pdb_id=pdb_id,
                structure_path=structure_path,
            )

            if len(df_one) == 0:
                failed_structures.append((pdb_id, "empty dataframe"))
                print("  empty dataframe", flush=True)
                continue

            df_one = add_amino_acid_properties(df_one)
            buffer.append(df_one)

        except Exception as e:
            failed_structures.append((pdb_id, repr(e)))
            print(f"  failed: {repr(e)}", flush=True)
            continue

        should_flush = len(buffer) >= chunk_size or (i + 1 == len(pdb_ids))

        if should_flush and len(buffer) > 0:
            chunk_df = pd.concat(buffer, ignore_index=True)

            chunk_df.to_csv(
                output_path,
                mode="w" if is_first_write else "a",
                header=is_first_write,
                index=False,
            )

            print(f"  wrote {len(chunk_df)} rows", flush=True)

            buffer = []
            is_first_write = False

    return missing_files, failed_structures

In [29]:
output_path = PROCESSED_DIR / "chi1_residue_level_dataset_full_features.csv"

missing_files, failed_structures = collect_full_feature_dataset_to_csv(
    pdb_ids=pdb_ids,
    output_path=output_path,
    chunk_size=100,
)

print("Saved to:", output_path)
print("Missing files:", len(missing_files))
print("Failed structures:", len(failed_structures))

[1/6108] 2BO5
[2/6108] 2BW2
[3/6108] 2IVW
[4/6108] 4ME2
[5/6108] 4MEI
[6/6108] 4MFI
[7/6108] 4MFJ
[8/6108] 4MGS
[9/6108] 4MHP
[10/6108] 4MPL
[11/6108] 4MQ3
[12/6108] 4MT7
[13/6108] 3PG4
[14/6108] 3PHS
[15/6108] 3PR9
[16/6108] 3PRD
[17/6108] 3PTD
[18/6108] 3PTE
[19/6108] 3PTW
[20/6108] 1Q5Z
[21/6108] 1Q7X
[22/6108] 1Q89
[23/6108] 1Q8G
[24/6108] 1Q8K
[25/6108] 1Q8X
[26/6108] 1Q9F
[27/6108] 1Q9G
[28/6108] 1QAU
[29/6108] 1QCX
[30/6108] 1QK8
[31/6108] 1QLP
[32/6108] 1QLX
[33/6108] 1QLZ
[34/6108] 1QM0
[35/6108] 1QM1
[36/6108] 1QM2
[37/6108] 1QM3
[38/6108] 1QM9
[39/6108] 1QMT
[40/6108] 1QND
[41/6108] 2K9N
[42/6108] 1X6Y
[43/6108] 2MC3
[44/6108] 5HIH
[45/6108] 5I7E
[46/6108] 5JO8
[47/6108] 5MMU
[48/6108] 1UJD
[49/6108] 1UJO
[50/6108] 1UJR
[51/6108] 1UJT
[52/6108] 1UJU
[53/6108] 1UJX
[54/6108] 1UK5
[55/6108] 1UKF
[56/6108] 1UKX
[57/6108] 1UL7
[58/6108] 1ULO
[59/6108] 1ULP
[60/6108] 1ULZ
[61/6108] 1UM1
[62/6108] 1UM7
[63/6108] 1UN3
[64/6108] 1UOK
[65/6108] 1UOR
[66/6108] 1UOT
[67/6108] 1UWD
[68/

In [30]:
full_df = pd.read_csv(PROCESSED_DIR / "chi1_residue_level_dataset_full_features.csv")

full_df.shape

/var/folders/j3/qr5mg5rj55j5lf7yh978r__80000gn/T/ipykernel_7356/4213164098.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  full_df = pd.read_csv(PROCESSED_DIR / "chi1_residue_level_dataset_full_features.csv")


(1165673, 42)

In [31]:
print("target_mask:")
print(full_df["target_mask"].value_counts())

print("\nhas_ca:")
print(full_df["has_ca"].value_counts())

print("\nhas_cb:")
print(full_df["has_cb"].value_counts())

print("\ndssp_has:")
print(full_df["dssp_has"].value_counts())

print("\ndssp_ss:")
print(full_df["dssp_ss"].value_counts())

target_mask:
target_mask
1    973654
0    192019
Name: count, dtype: int64

has_ca:
has_ca
1    1165673
Name: count, dtype: int64

has_cb:
has_cb
1    1069905
0      95768
Name: count, dtype: int64

dssp_has:
dssp_has
0    1163925
1       1748
Name: count, dtype: int64

dssp_ss:
dssp_ss
UNK    1163925
H          593
E          349
C          329
T          190
S          166
G           70
P           29
B           17
I            5
Name: count, dtype: int64


In [32]:
full_df["dssp_has"].value_counts()

dssp_has
0    1163925
1       1748
Name: count, dtype: int64